In [ ]:
import pandas as pd
import altair as alt

# 1. Load the dataset from the local directory
df = pd.read_csv('uber_dataset.csv')

# 2. Convert 'Time' column to datetime objects and extract the 'Hour'
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
df['Hour'] = df['Time'].dt.hour

# 3. Filter out rows with missing values in the crucial columns
df_clean = df.dropna(subset=['Ride Distance', 'Booking Value'])

print("Block 1 completed: Data loaded and cleaned successfully.")

C:\Users\DELL\AppData\Local\Temp\ipykernel_26432\2366252641.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time'], errors='coerce')


Block 1 completed: Data loaded and cleaned successfully.


In [ ]:
# 1. Define the Semantic Structure for Generalised Selection
def get_time_of_day(hour):
    if pd.isna(hour):
        return 'Unknown'
    elif 6 <= hour < 12: 
        return 'Morning (6-11)'
    elif 12 <= hour < 18: 
        return 'Afternoon (12-17)'
    elif 18 <= hour < 24: 
        return 'Evening (18-23)'
    else: 
        return 'Night (0-5)'

# Apply the hierarchy to the dataset
df_clean['Time_of_Day'] = df_clean['Hour'].apply(get_time_of_day)

# 2. Sample data for smooth interactive rendering
# Taking 5000 rows ensures the output html file is lightweight and doesnt lag
df_sample = df_clean.sample(n=5000, random_state=42)

print("Block 2 completed: Semantic structure applied and data sampled.")

Block 2 completed: Semantic structure applied and data sampled.


C:\Users\DELL\AppData\Local\Temp\ipykernel_26432\1041233666.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Time_of_Day'] = df_clean['Hour'].apply(get_time_of_day)


In [ ]:
# 1. Define Interactive Selections
# Level 0 selection standard brushing
brush = alt.selection_interval(encodings=['x', 'y'])
# Level 1 selection clicking to traverse up the semantic tree
generalise_selection = alt.selection_point(fields=['Time_of_Day'])

# 2. View 1 - 2D Binned Heatmap
heatmap = alt.Chart(df_sample).mark_rect().encode(
    x=alt.X('Ride Distance:Q', bin=alt.Bin(maxbins=40), title='Ride Distance (km)'),
    y=alt.Y('Booking Value:Q', bin=alt.Bin(maxbins=40), title='Booking Value (₹)'),
    color=alt.condition(
        generalise_selection, 
        alt.Color('count():Q', scale=alt.Scale(scheme='greenblue'), title='Rides'),
        alt.value('lightgray')
    ),
    tooltip=['count()']
).add_params(
    brush
).properties(width=400, height=300, title="View 1: Distance vs. Value Density")

# 3. View 2- Vehicle Type Distribution
vehicle_chart = alt.Chart(df_sample).mark_bar().encode(
    x=alt.X('Vehicle Type:N', sort='-y'),
    y=alt.Y('count():Q', title='Number of Rides'),
    color='Vehicle Type:N'
).transform_filter(
    brush
).transform_filter(
    generalise_selection
).properties(
    width=400,
    height=250,
    title="Vehicle Type Distribution"
)


# 4. View 3 - Stacked Bar Chart
bar_chart = alt.Chart(df_sample).mark_bar().encode(
    x=alt.X('Hour:O', title='Hour of Day'),
    y=alt.Y('count():Q', title='Number of Rides'),
    color=alt.Color('Booking Status:N', scale=alt.Scale(scheme='tableau10'))
).transform_filter(
    brush
).transform_filter(
    generalise_selection
).properties(width=400, height=300, title="View 2: Hourly Status Breakdown")

# 5. View 3 - Generalisation Control Panel
generalise_control = alt.Chart(df_sample).mark_bar().encode(
    y=alt.Y('Time_of_Day:N', title='Generalise to Time of Day', sort=['Morning (6-11)', 'Afternoon (12-17)', 'Evening (18-23)', 'Night (0-5)']),
    x=alt.X('count():Q', title='Total Rides'),
    color=alt.condition(
        generalise_selection, 
        alt.value('coral'), 
        alt.value('lightgray')
    )
).add_params(
    generalise_selection
).properties(width=850, height=80, title="Generalised Selection Control: Click to select whole time blocks")

# 6. Combine all views
# Put Heatmap and Bar Chart side by side, then put Control Panel below them
system_b = ((heatmap | bar_chart) & vehicle_chart) & generalise_control

# 7. EXPORT TO HTMl
html_filename = 'System_B_Interactive.html'
system_b.save(html_filename)
print(f"🎉 Success! The interactive dashboard has been saved as '{html_filename}' in your current folder.")

# Render in the notebook as well
system_b

🎉 Success! The interactive dashboard has been saved as 'System_B_Interactive.html' in your current folder.


alt.VConcatChart(...)